In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from scipy.stats import wilcoxon

TABLES_DIR = Path(r"C:\Users\HUGO\Desktop\Q8 - NORUEGA\TFG\tfg\results\final_event_training_experiment\final_run\tables")

RESULTS_PATH = TABLES_DIR / "all_week_results.csv"

OUTPUT_DIR = TABLES_DIR / "statistical_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(RESULTS_PATH)

print(df.shape)
df.head()

(450, 18)


,Zone,Calibration,Calibration period,Test week,Test period,Model,Dataset,Training samples,Test samples,price_values,buy_volume_values,sell_volume_values,flow_values,capacity_values,total_raw_values,Raw values used (%),Raw value reduction (%),RMSE
0,NO2,cal_2020,2020-01-01 to 2020-12-31,winter_january,2024-01-15 to 2024-01-21,Linear Regression,full_dataset,8764,168,8760,8760,8760,22548,17518,66346,100.000000,0.000000,7.464502
1,NO2,cal_2020,2020-01-01 to 2020-12-31,winter_january,2024-01-15 to 2024-01-21,Linear Regression,own_events,8764,168,8761,6266,3233,8403,1310,27973,42.162301,57.837699,7.464502
2,NO2,cal_2020,2020-01-01 to 2020-12-31,winter_january,2024-01-15 to 2024-01-21,Linear Regression,rbatheta_events,5696,168,2212,1101,2394,9189,0,14896,22.451994,77.548006,7.367459
3,NO2,cal_2020,2020-01-01 to 2020-12-31,winter_january,2024-01-15 to 2024-01-21,Decision Tree,full_dataset,8764,168,8760,8760,8760,22548,17518,66346,100.000000,0.000000,12.471897
4,NO2,cal_2020,2020-01-01 to 2020-12-31,winter_january,2024-01-15 to 2024-01-21,Decision Tree,own_events,8764,168,8761,6266,3233,8403,1310,27973,42.162301,57.837699,12.471897


In [4]:
df.columns

Index(['Zone', 'Calibration', 'Calibration period', 'Test week', 'Test period',
       'Model', 'Dataset', 'Training samples', 'Test samples', 'price_values',
       'buy_volume_values', 'sell_volume_values', 'flow_values',
       'capacity_values', 'total_raw_values', 'Raw values used (%)',
       'Raw value reduction (%)', 'RMSE'],
      dtype='str')

In [5]:
results = df.copy()

required_cols = ["Zone", "Calibration", "Test week", "Model", "Dataset", "RMSE"]

missing_cols = [col for col in required_cols if col not in results.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

results["RMSE"] = pd.to_numeric(results["RMSE"], errors="coerce")
results = results.dropna(subset=["RMSE"]).copy()

print("Zones:", results["Zone"].unique())
print("Calibrations:", results["Calibration"].unique())
print("Datasets:", results["Dataset"].unique())
print("Models:", results["Model"].unique())
print("Test weeks:", results["Test week"].unique())
print("Rows:", len(results))

Zones: <StringArray>
['NO2', 'NO4']
Length: 2, dtype: str
Calibrations: <StringArray>
['cal_2020', 'cal_2020_2021', 'cal_2020_2022']
Length: 3, dtype: str
Datasets: <StringArray>
['full_dataset', 'own_events', 'rbatheta_events']
Length: 3, dtype: str
Models: <StringArray>
['Linear Regression', 'Decision Tree', 'Random Forest', 'XGBoost', 'ARIMA']
Length: 5, dtype: str
Test weeks: <StringArray>
[  'winter_january',     'spring_april',      'summer_june',
 'autumn_september',  'winter_december']
Length: 5, dtype: str
Rows: 450


In [6]:
def add_confidence_interval(table, mean_col="RMSE_mean", std_col="RMSE_std", n_col="n"):
    table = table.copy()
    
    table["t_value"] = table[n_col].apply(
        lambda n: stats.t.ppf(0.975, n - 1) if n > 1 else np.nan
    )
    
    table["CI95_margin"] = table["t_value"] * table[std_col] / np.sqrt(table[n_col])
    table["CI95_lower"] = table[mean_col] - table["CI95_margin"]
    table["CI95_upper"] = table[mean_col] + table["CI95_margin"]
    
    return table.drop(columns=["t_value"])

# Average across models for each week
weekly_avg = (
    results
    .groupby(["Zone", "Calibration", "Dataset", "Test week"], as_index=False)
    .agg(
        RMSE=("RMSE", "mean"),
        total_raw_values=("total_raw_values", "first")
    )
)

# Statistical summary across the five test weeks
summary_compact = (
    weekly_avg
    .groupby(["Zone", "Calibration", "Dataset"], as_index=False)
    .agg(
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        n=("RMSE", "count"),
        total_raw_values=("total_raw_values", "first")
    )
)

summary_compact = add_confidence_interval(summary_compact)

summary_compact = summary_compact.sort_values(
    ["Zone", "Calibration", "Dataset"]
).reset_index(drop=True)

summary_compact.to_csv(OUTPUT_DIR / "statistical_validation_compact.csv", index=False)
summary_compact.to_excel(OUTPUT_DIR / "statistical_validation_compact.xlsx", index=False)

summary_compact

,Zone,Calibration,Dataset,RMSE_mean,RMSE_std,n,total_raw_values,CI95_margin,CI95_lower,CI95_upper
0,NO2,cal_2020,full_dataset,18.383202,5.168628,5,66346,6.417699,11.965503,24.800901
1,NO2,cal_2020,own_events,18.383202,5.168628,5,27973,6.417699,11.965503,24.800901
2,NO2,cal_2020,rbatheta_events,18.639996,5.588300,5,14896,6.938791,11.701205,25.578788
3,NO2,cal_2020_2021,full_dataset,18.383202,5.168628,5,66346,6.417699,11.965503,24.800901
4,NO2,cal_2020_2021,own_events,18.501696,5.206264,5,23435,6.464431,12.037266,24.966127
5,NO2,cal_2020_2021,rbatheta_events,18.639996,5.588300,5,14896,6.938791,11.701205,25.578788
6,NO2,cal_2020_2022,full_dataset,18.383202,5.168628,5,66346,6.417699,11.965503,24.800901
7,NO2,cal_2020_2022,own_events,18.574854,5.386475,5,14967,6.688192,11.886661,25.263046
8,NO2,cal_2020_2022,rbatheta_events,18.639996,5.588300,5,14896,6.938791,11.701205,25.578788
9,NO4,cal_2020,full_dataset,6.818543,4.106495,5,119313,5.098888,1.719656,11.917431


In [7]:
wilcoxon_results = []

for zone in weekly_avg["Zone"].unique():
    for calibration in weekly_avg["Calibration"].unique():
        
        subset = weekly_avg[
            (weekly_avg["Zone"] == zone) &
            (weekly_avg["Calibration"] == calibration)
        ].copy()
        
        full = subset[subset["Dataset"] == "full_dataset"]
        
        if full.empty:
            continue
        
        for dataset in subset["Dataset"].unique():
            
            if dataset == "full_dataset":
                continue
            
            event_data = subset[subset["Dataset"] == dataset]
            
            merged = full.merge(
                event_data,
                on=["Zone", "Calibration", "Test week"],
                suffixes=("_full", "_event")
            )
            
            if len(merged) < 2:
                continue
            
            try:
                stat, p_value = wilcoxon(
                    merged["RMSE_full"],
                    merged["RMSE_event"],
                    zero_method="wilcox",
                    alternative="two-sided"
                )
            except ValueError:
                stat, p_value = np.nan, np.nan
            
            wilcoxon_results.append({
                "Zone": zone,
                "Calibration": calibration,
                "Comparison": f"full_dataset vs {dataset}",
                "n": len(merged),
                "RMSE_full_mean": merged["RMSE_full"].mean(),
                "RMSE_event_mean": merged["RMSE_event"].mean(),
                "Mean_difference_event_minus_full": (
                    merged["RMSE_event"] - merged["RMSE_full"]
                ).mean(),
                "Wilcoxon_statistic": stat,
                "p_value": p_value
            })

wilcoxon_table = pd.DataFrame(wilcoxon_results)

wilcoxon_table = wilcoxon_table.sort_values(
    ["Zone", "Calibration", "Comparison"]
).reset_index(drop=True)

wilcoxon_table.to_csv(OUTPUT_DIR / "wilcoxon_results_compact.csv", index=False)
wilcoxon_table.to_excel(OUTPUT_DIR / "wilcoxon_results_compact.xlsx", index=False)

wilcoxon_table

c:\Users\HUGO\AppData\Local\Programs\Python\Python312\Lib\site-packages\scipy\stats\_wilcoxon.py:181: RuntimeWarning: invalid value encountered in scalar divide
  z = (r_plus - mn) / se


,Zone,Calibration,Comparison,n,RMSE_full_mean,RMSE_event_mean,Mean_difference_event_minus_full,Wilcoxon_statistic,p_value
0,NO2,cal_2020,full_dataset vs own_events,5,18.383202,18.383202,0.000000,0.0,1.0000
1,NO2,cal_2020,full_dataset vs rbatheta_events,5,18.383202,18.639996,0.256794,6.0,0.8125
2,NO2,cal_2020_2021,full_dataset vs own_events,5,18.383202,18.501696,0.118494,3.0,0.3125
3,NO2,cal_2020_2021,full_dataset vs rbatheta_events,5,18.383202,18.639996,0.256794,6.0,0.8125
4,NO2,cal_2020_2022,full_dataset vs own_events,5,18.383202,18.574854,0.191651,5.0,0.6250
5,NO2,cal_2020_2022,full_dataset vs rbatheta_events,5,18.383202,18.639996,0.256794,6.0,0.8125
6,NO4,cal_2020,full_dataset vs own_events,5,6.818543,6.813673,-0.004870,5.0,0.6250
7,NO4,cal_2020,full_dataset vs rbatheta_events,5,6.818543,9.707237,2.888694,0.0,0.0625
8,NO4,cal_2020_2021,full_dataset vs own_events,5,6.818543,6.820905,0.002362,6.0,0.8125
9,NO4,cal_2020_2021,full_dataset vs rbatheta_events,5,6.818543,9.707237,2.888694,0.0,0.0625
